# G1 — The shared space as an object: one hub, five encoders, zero-shot
# component transfer

Every result so far is *pairwise*: encoder A relates to encoder B by a
linear map. N encoders would need N-squared maps, and each map is fitted
against its own target, so nothing yet shows that the encoders share
**one** coordinate system rather than a web of bilateral agreements.

This notebook tests the stronger claim. A single hub space is
constructed — a whitened PCA basis, fitted on training rows of one
reference encoder — and every encoder is given one linear map into it.
Two questions follow, and the second is the substantive one.

**1. Transitivity.** Fit A→hub and hub→B, never fitting A→B. Does the
composition match a directly fitted A→B? If the hub were an arbitrary
intermediate, composing two lossy maps would degrade badly. If the
encoders genuinely share a coordinate system, the composition should
approach the direct fit.

**2. Zero-shot component transfer.** Train a caption-prediction head on
hub vectors from **one** encoder only. Then feed a **different**
encoder's data through its own hub map into the same head. The head has
never seen that encoder, at any point, in any form. If retrieval still
works, a component built in the hub is portable across encoders that
were never trained together — which is the shared space demonstrated by
use rather than by correlation.

**Pre-registered reading.** Composition within 15 per cent of the direct
fit counts as transitivity holding. Transfer retaining at least half the
native head's R@1 counts as portability; below a quarter counts as
failure. A random-map control must sit at chance, or the measurement is
meaningless.

**Data: nothing new is encoded.** The DINOv2 sweep used nested prefixes
of the same sorted COCO id list, so the three image caches share their
first 9,533 rows, and the E1/E1.1 pairs files were built over the same
ordering. Alignment is verified numerically below rather than
assumed.

In [ ]:
import os
from pathlib import Path
STORAGE   = "drive"                     # "drive" | "local" | "env"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"
try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR is unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    if STORAGE == "drive":
        print("not on Colab - using LOCAL_DIR")
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())
DATA_DIR = Path(os.environ["DATA_DIR"])
print("DATA_DIR:", DATA_DIR)

In [ ]:
import numpy as np

# ---- gather every cached space, and VERIFY they are row-aligned ----
SPACES, N_COMMON = {}, None
for size in ("small", "base", "large"):
    f = DATA_DIR / f"e1_img_ckpt_dinov2-{size}_cls+patch.npz"
    if f.exists():
        d = np.load(str(f))
        SPACES[f"img_{size}"] = d["img"].astype(np.float64)
        n = len(d["img"])
        N_COMMON = n if N_COMMON is None else min(N_COMMON, n)
        print(f"  img_{size:5s} {d['img'].shape}")
assert len(SPACES) >= 2, "need at least two image caches - run E1 first"

def add_text(candidates, key):
    """Attach a text space from whichever cached pairs file is present.

    The pairs files carry no ids, so alignment is PROVEN rather than
    assumed: the file's image block must match one of the image caches
    already loaded. Several filenames are tried because early runs were
    saved before the encoder size was added to the name - and for a text
    encoder the image side is irrelevant anyway, since captions are
    encoded independently of it."""
    for fname in candidates:
        f = DATA_DIR / fname
        if not f.exists():
            continue
        d = np.load(str(f))
        for ref in [k for k in SPACES if k.startswith("img_")]:
            # width must match before values can be compared at all
            if d["img"].shape[1] != SPACES[ref].shape[1]:
                continue
            n = min(len(d["img"]), len(SPACES[ref]))
            if n < 500:
                continue
            if np.allclose(d["img"][:n], SPACES[ref][:n], atol=1e-4):
                SPACES[key] = d["txt"].astype(np.float64)[:n]
                print(f"  {key:11s} {SPACES[key].shape}  from {fname} "
                      f"(alignment verified against {ref})")
                return
        print(f"  ({fname} present but matches no image cache - skipped)")
    print(f"  (no usable file for {key}: tried {', '.join(candidates)})")

add_text(["crossmodal_pairs.npz"], "txt_bge")
add_text(["crossmodal_pairs_gpt2.npz",                       # pre-rename
          "crossmodal_pairs_gpt2_dinov2-base_cls+patch.npz",
          "crossmodal_pairs_gpt2_dinov2-large_cls+patch.npz",
          "crossmodal_pairs_gpt2_dinov2-small_cls+patch.npz"], "txt_gpt2")

N_COMMON = min(min(len(v) for v in SPACES.values()), N_COMMON)
SPACES = {k: v[:N_COMMON] for k, v in SPACES.items()}
print(f"\n{len(SPACES)} spaces on {N_COMMON} common items: "
      f"{list(SPACES)}")

rng = np.random.default_rng(0)
perm = rng.permutation(N_COMMON)
N_EVAL = 1000
te, tr = perm[:N_EVAL], perm[N_EVAL:]
print(f"{len(tr)} train / {len(te)} eval")
for k, v in SPACES.items():
    print(f"   {k:11s} width {v.shape[1]:5d}  "
          f"{len(tr)/v.shape[1]:5.1f} rows per input dim")

## Building the hub

The hub is a whitened PCA basis of one reference encoder, estimated on
training rows only. Whitening is used because Section C.11 showed that
an isotropic target is what makes cosine readable — the hub is the space
components will be *read in*, so it should be isotropic. (Section C.12
showed the opposite prescription applies to *discovering* an unknown
correspondence; that is not what happens here, since every map is fitted
on known rows.)

In [ ]:
# A 256-d hub cannot RECONSTRUCT a 768- or 1024-d target: measured on
# the Experiment B spaces, retention into the widest target went from
# -444% at 256 to +84% at 2048. Reconstruction needs width; the transfer
# test below does not, and passed even at 256. Sweep this against the
# rows-per-hub-dim ratio printed above - more width is not free.
HUB_DIM  = 1024
ALPHA    = 1e-2

# HUB_MODE decides what the hub is a basis OF, and it matters:
#   "single"   - whitened PCA of one reference encoder. Cheap, but the
#                hub then represents that encoder's modality well and
#                others poorly, so maps INTO the under-represented
#                spaces lose more than they should.
#   "balanced" - whitened PCA of every space concatenated, each first
#                standardised so no space dominates by width or scale.
#                Costs nothing extra and removes the blind spot.
HUB_MODE = "balanced"            # "balanced" | "single"
HUB_REF  = "img_base" if "img_base" in SPACES else list(SPACES)[0]

def l2n(V):
    return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)
def ridge(X, Y, a=ALPHA):
    return np.linalg.solve(X.T @ X + a * np.eye(X.shape[1]), X.T @ Y)
def r2(Y, P):
    return float(1 - ((Y - P) ** 2).sum() / ((Y - Y.mean(0)) ** 2).sum())
def recall(S):
    o = np.argsort(-S, 1); r = (o == np.arange(len(S))[:, None]).argmax(1)
    return {k: float((r < k).mean()) for k in (1, 5, 10)}

if HUB_MODE == "single":
    ref = SPACES[HUB_REF][tr]
else:
    # standardise each space before concatenating, so a wider or
    # higher-variance space cannot dominate the shared basis
    parts = []
    for k in SPACES:
        v = SPACES[k][tr]
        parts.append((v - v.mean(0)) / (v.std(0).mean() + 1e-12))
    ref = np.hstack(parts)
    print(f"balanced hub over {len(parts)} spaces, "
          f"concatenated width {ref.shape[1]}")

mu = ref.mean(0)
_U, _s, _Vt = np.linalg.svd(ref - mu, full_matrices=False)
K = min(HUB_DIM, len(_s))
BASIS = _Vt[:K].T / (_s[:K] / np.sqrt(len(ref)))          # whitened PCA
HUB_TR = (ref - mu) @ BASIS            # hub coordinates of the TRAIN rows

def hub_coords(which):
    """Hub coordinates for train/eval rows, built the same way for both."""
    if HUB_MODE == "single":
        return (SPACES[HUB_REF][which] - mu) @ BASIS
    parts = []
    for k in SPACES:
        v = SPACES[k][which]
        vt = SPACES[k][tr]
        parts.append((v - vt.mean(0)) / (vt.std(0).mean() + 1e-12))
    return (np.hstack(parts) - mu) @ BASIS

HUB = {"tr": HUB_TR, "te": hub_coords(te)}

# one map per encoder INTO the hub - fitted on train rows only
TO_HUB = {k: ridge(v[tr], HUB["tr"]) for k, v in SPACES.items()}
print(f"hub: {K}-d whitened PCA, mode={HUB_MODE}")
print("   how well each space can REACH the hub (held-out R2):")
for k in SPACES:
    print(f"   {k:11s} {r2(HUB['te'], SPACES[k][te] @ TO_HUB[k]):6.3f}")
print("   a low value here predicts poor reconstruction INTO that space")

## Test 1 — transitivity: composition through the hub vs a direct fit

In [ ]:
names = list(SPACES)
print(f"{'A -> B':26s} {'direct R2':>10s} {'via hub':>9s} {'retained':>9s}")
rows = []
for a in names:
    for b in names:
        if a == b: continue
        direct = ridge(SPACES[a][tr], SPACES[b][tr])
        from_hub = ridge(SPACES[a][tr] @ TO_HUB[a], SPACES[b][tr])
        rd = r2(SPACES[b][te], SPACES[a][te] @ direct)
        rh = r2(SPACES[b][te], (SPACES[a][te] @ TO_HUB[a]) @ from_hub)
        if rd > 0.05:
            ratio = rh / rd
            rows.append((a, b, rd, rh, ratio))
            # a near-zero direct R2 makes the ratio explode; flag rather
            # than print a meaningless six-digit percentage
            shown = (f"{100*ratio:8.1f}%" if rd > 0.15
                     else "   n/a *")
            print(f"{a+' -> '+b:26s} {rd:10.3f} {rh:9.3f} {shown}")
    # * ratio omitted where the direct fit itself is near zero (<0.15),
    #   since a tiny denominator makes the percentage uninformative

# retention depends on how well the TARGET is represented in the hub,
# so report it against that rather than as one average over everything
print(f"\n{'target space':14s} {'hub R2':>8s} {'mean retention':>15s}")
for t in SPACES:
    sub = [r[4] for r in rows if r[1] == t]
    if sub:
        hr = r2(HUB["te"], SPACES[t][te] @ TO_HUB[t])
        print(f"{t:14s} {hr:8.3f} {100*np.mean(sub):14.1f}%")
# average only over targets the hub can actually represent (direct
# fit above 0.15), so one blow-up does not dominate the summary
allr = [r[4] for r in rows if r[2] > 0.15]
print(f"\nmean retention (targets with direct R2 > 0.15): "
      f"{100*np.mean(allr):.1f}%  "
      f"(pre-registered: >= 85%)")
print("targets the hub represents poorly cannot be reconstructed from it;"
      "\nthat is a capacity property of the hub, not evidence about the"
      "\nencoders - raise HUB_DIM or use HUB_MODE='balanced' and re-run.")

## Choosing HUB_DIM by measurement

`HUB_DIM` trades two things against each other. Too narrow and the hub
cannot RECONSTRUCT a wide target: on the Experiment B spaces, retention
into the widest target ran at -444 per cent with a 256-d hub and +84 per
cent at 2048. Too wide and the maps into the hub become underpowered -
rows per hub dimension falls, and this project's own threshold is five.

The sweep below reports both, plus the transfer result, so the choice is
made from evidence rather than argued. The singular value decomposition
is computed once and sliced, so additional widths are nearly free.

Verification AUC is included alongside retrieval. It asks the weaker
question of Appendix D.2 - can a threshold separate a true pair from a
random one - and a transferred component may hold up far better there
than its R@1 suggests. That is supplementary evidence, not the headline.

In [ ]:
# ---- HUB_DIM x ALPHA sweep, transfer-first ----
# The transfer test wants a NARROW hub (whitened PCA amplifies tail
# noise directions, which dominate cosine geometry past a threshold);
# reconstruction wants a WIDE one. They are not reconcilable, and that
# is the finding. Alpha shifts the cliff slightly but does not move it
# past the narrow setting, so this grid exists to LOCATE the cliff on
# your data, not to defeat it.
if HUB_MODE == "single":
    _ref = SPACES[HUB_REF][tr]
else:
    _ref = np.hstack([(SPACES[k][tr] - SPACES[k][tr].mean(0)) /
                      (SPACES[k][tr].std(0).mean() + 1e-12)
                      for k in SPACES])
_mu = _ref.mean(0)
_U, _sv, _VT = np.linalg.svd(_ref - _mu, full_matrices=False)

def _co(which, K):
    B = _VT[:K].T / (_sv[:K] / np.sqrt(len(_ref)))
    if HUB_MODE == "single":
        return (SPACES[HUB_REF][which] - _mu) @ B
    X = np.hstack([(SPACES[k][which] - SPACES[k][tr].mean(0)) /
                   (SPACES[k][tr].std(0).mean() + 1e-12) for k in SPACES])
    return (X - _mu) @ B

def _auc(pos, neg):
    y = np.r_[np.ones(len(pos)), np.zeros(len(neg))]; s = np.r_[pos, neg]
    o = np.argsort(-s); y = y[o]
    return float(np.trapezoid(np.cumsum(y)/y.sum(),
                              np.cumsum(1-y)/(1-y).sum()))

_don = [k for k in SPACES if k.startswith("img_")]
assert "txt_bge" in SPACES and len(_don) > 1, "need txt_bge + 2 image spaces"
_gal = l2n(SPACES["txt_bge"][te])
_rng = np.random.default_rng(1)
_j = _rng.permutation(len(te))
_j = np.where(_j == np.arange(len(te)), (_j + 1) % len(te), _j)

DIMS   = [k for k in (128, 256, 512, 768, 1024) if k <= len(_sv)]
ALPHAS = [1e-2, 1e-1, 1.0, 10.0]

print("TRANSFER R@1  (head trained on", _don[0], "-> unseen", _don[1] + ")")
print(f"{'HUB_DIM':>8s} {'rows/dim':>9s}" +
      "".join(f"{'a='+str(a):>9s}" for a in ALPHAS))
best = (-1, None, None)
grid = {}
for K in DIMS:
    htr, hte = _co(tr, K), _co(te, K)
    line = f"{K:8d} {len(tr)/K:9.1f}"
    for a in ALPHAS:
        TH = {k: np.linalg.solve(SPACES[k][tr].T @ SPACES[k][tr] +
                                 a*np.eye(SPACES[k].shape[1]),
                                 SPACES[k][tr].T @ htr) for k in SPACES}
        hd = np.linalg.solve((SPACES[_don[0]][tr]@TH[_don[0]]).T @
                             (SPACES[_don[0]][tr]@TH[_don[0]]) +
                             a*np.eye(K),
                             (SPACES[_don[0]][tr]@TH[_don[0]]).T @
                             SPACES["txt_bge"][tr])
        P = l2n((SPACES[_don[1]][te] @ TH[_don[1]]) @ hd)
        r1 = recall(P @ _gal.T)[1]
        grid[(K, a)] = (r1, _auc((P*_gal).sum(1), (P*_gal[_j]).sum(1)))
        line += f"{r1:9.3f}"
        if r1 > best[0] and len(tr)/K >= 5:      # respect the rows/dim floor
            best = (r1, K, a)
    print(line)
print(f"\nbest transfer with rows/dim >= 5: "
      f"R@1 {best[0]:.3f} at HUB_DIM={best[1]}, ALPHA={best[2]} "
      f"(verif AUC {grid[(best[1],best[2])][1]:.3f})")
print("set HUB_DIM and ALPHA in the config cell to these, then re-run the")
print("transfer cell for the headline numbers.")

## Test 2 — zero-shot transfer of a trained component

The head maps hub vectors to the bge-m3 caption space and is trained on
**one** encoder's hub vectors only. It is then applied to other encoders'
hub vectors and scored by caption retrieval on the same held-out rows.
Two controls: the native head fitted directly on each encoder (an upper
reference), and the same head fed through a random map (which must sit
at chance, or the test proves nothing).

In [ ]:
assert "txt_bge" in SPACES, "needs the bge-m3 space - run E1 first"
TARGET = "txt_bge"
donors = [k for k in SPACES if k.startswith("img_")]
TRAIN_ON = donors[0]                      # smallest encoder available

head = ridge(SPACES[TRAIN_ON][tr] @ TO_HUB[TRAIN_ON],
             SPACES[TARGET][tr])
gal = l2n(SPACES[TARGET][te])
print(f"head trained ONLY on {TRAIN_ON}, applied through the hub\n")
print(f"{'encoder':12s} {'R@1':>7s} {'R@5':>7s} {'R@10':>7s} "
      f"{'vs native R@1':>14s}")
for enc in donors:
    P = l2n((SPACES[enc][te] @ TO_HUB[enc]) @ head)
    r = recall(P @ gal.T)
    nat = ridge(SPACES[enc][tr], SPACES[TARGET][tr])
    rn = recall(l2n(SPACES[enc][te] @ nat) @ gal.T)
    tag = "  <- trained here" if enc == TRAIN_ON else "  <- NEVER SEEN"
    print(f"{enc:12s} {r[1]:7.3f} {r[5]:7.3f} {r[10]:7.3f} "
          f"{100*r[1]/max(rn[1],1e-9):13.1f}%{tag}")

Rrand = rng.standard_normal(TO_HUB[donors[-1]].shape) / \
        np.sqrt(SPACES[donors[-1]].shape[1])
rr = recall(l2n((SPACES[donors[-1]][te] @ Rrand) @ head) @ gal.T)
print(f"\ncontrol - random map into the hub: R@1 {rr[1]:.3f} "
      f"(chance {1/N_EVAL:.3f})")

## How to read this

Transitivity says the hub is not an arbitrary waypoint: if composing
A→hub→B retains most of a direct A→B fit, the encoders are being
described in one coordinate system rather than by a web of bilateral
agreements.

Zero-shot transfer is the constructive claim. A head trained on one
encoder's hub vectors, applied to a different encoder that it never saw
in any form, is a working system assembled from parts that were never
trained together — the shared space demonstrated by use.

Report the retention percentages, not just whether it "works", and
report the random-map control alongside: without it, a transfer number
means nothing. Note also the scope — the encoders here come from two
families on one image domain, so this exhibits a shared space for these
five, not for all models.